In [6]:
# This cell sets correct folder paths.

from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = REPO_ROOT / "Dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSON_FILE = DATA_DIR / "ai_papers_raw_15k.json"
CSV_FILE = DATA_DIR / "ai_papers_clean_15k.csv"

print("Notebook folder:", NOTEBOOK_DIR)
print("Project root:", REPO_ROOT)
print("Dataset folder:", DATA_DIR)
print("Raw JSON output:", RAW_JSON_FILE)
print("Clean CSV output:", CSV_FILE)

Notebook folder: c:\Users\Nguyen Phuong Thao\Project_2_Data_Visualization_Group_6\Notebooks
Project root: c:\Users\Nguyen Phuong Thao\Project_2_Data_Visualization_Group_6
Dataset folder: c:\Users\Nguyen Phuong Thao\Project_2_Data_Visualization_Group_6\Dataset
Raw JSON output: c:\Users\Nguyen Phuong Thao\Project_2_Data_Visualization_Group_6\Dataset\ai_papers_raw_15k.json
Clean CSV output: c:\Users\Nguyen Phuong Thao\Project_2_Data_Visualization_Group_6\Dataset\ai_papers_clean_15k.csv


In [7]:
# OpenAlex asks for an email for polite API usage.
# Use your own email here.

OPENALEX_EMAIL = "phuongthao.2006894@gmail.com"

print("OpenAlex email:", OPENALEX_EMAIL)

OpenAlex email: phuongthao.2006894@gmail.com


In [8]:
# Import required libraries.
# If pyalex gives error, install it first:
# pip install pyalex pandas

import json
import time
import pandas as pd
import pyalex
from pyalex import Works

pyalex.config.email = OPENALEX_EMAIL

print("Libraries imported successfully.")
print("Using OpenAlex email:", pyalex.config.email)

Libraries imported successfully.
Using OpenAlex email: phuongthao.2006894@gmail.com


In [9]:
# Target number of papers.
# Your current dataset has around 7.4k rows.
# This crawler tries to collect 15k unique usable AI papers.

TARGET_RECORDS = 15000
PER_PAGE = 200
CURRENT_YEAR = 2026

# Multiple search terms help expand the dataset.
SEARCH_TERMS = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "large language model",
    "natural language processing",
    "computer vision",
    "reinforcement learning",
    "generative AI",
]

# Keep only useful academic publication types.
ALLOWED_TYPES = {
    "article",
    "preprint",
    "book-chapter",
    "proceedings-article",
}

print("Target records:", TARGET_RECORDS)
print("Search terms:", SEARCH_TERMS)

Target records: 15000
Search terms: ['artificial intelligence', 'machine learning', 'deep learning', 'large language model', 'natural language processing', 'computer vision', 'reinforcement learning', 'generative AI']


In [10]:
# These functions safely extract nested values from OpenAlex JSON.
# OpenAlex data has many nested dictionaries/lists, so direct access can cause errors.

def safe_get(d, keys, default=None):
    """
    Safely get nested dictionary value.
    Example:
    safe_get(paper, ["primary_topic", "field", "display_name"])
    """
    cur = d
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
        if cur is None:
            return default
    return cur


def safe_list(x):
    """
    Convert None to empty list.
    Keep list as list.
    Convert single value to list.
    """
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def join_unique(values):
    """
    Join unique non-empty values into one string.
    Example:
    ['USA', 'USA', 'China'] -> 'China; USA'
    """
    cleaned = [str(v) for v in safe_list(values) if v not in [None, ""]]
    return "; ".join(sorted(set(cleaned)))


print("Helper functions ready.")

Helper functions ready.


In [11]:
# This is the main crawling cell.
# It searches multiple AI keywords and removes duplicate papers by paper_id.

all_papers = {}
search_term_map = {}

print("Starting OpenAlex crawl...")

for term in SEARCH_TERMS:
    print(f"\nSearching term: {term}")

    query = (
        Works()
        .search(term)
        .filter(
            has_doi=True,
            from_publication_date="2000-01-01",
            to_publication_date="2026-12-31",
            is_retracted=False,
            is_paratext=False,
        )
        .sort(cited_by_count="desc")
    )

    page_count = 0

    for page in query.paginate(per_page=PER_PAGE):
        page_count += 1

        for paper in page:
            paper_id = paper.get("id")

            # Skip invalid paper records.
            if not paper_id:
                continue

            # Keep only selected academic publication types.
            if paper.get("type") not in ALLOWED_TYPES:
                continue

            # Deduplicate by OpenAlex paper ID.
            if paper_id not in all_papers:
                all_papers[paper_id] = paper
                search_term_map[paper_id] = term

            # Stop when target is reached.
            if len(all_papers) >= TARGET_RECORDS:
                break

        print(
            f"Term: {term} | Page: {page_count} | "
            f"Collected: {len(all_papers)} / {TARGET_RECORDS}"
        )

        if len(all_papers) >= TARGET_RECORDS:
            break

        # Small delay to avoid sending requests too aggressively.
        time.sleep(0.1)

    if len(all_papers) >= TARGET_RECORDS:
        break

all_papers = list(all_papers.values())[:TARGET_RECORDS]

print("\nFinished crawling.")
print("Total unique papers:", len(all_papers))

Starting OpenAlex crawl...

Searching term: artificial intelligence
Term: artificial intelligence | Page: 1 | Collected: 155 / 15000
Term: artificial intelligence | Page: 2 | Collected: 305 / 15000
Term: artificial intelligence | Page: 3 | Collected: 456 / 15000
Term: artificial intelligence | Page: 4 | Collected: 602 / 15000
Term: artificial intelligence | Page: 5 | Collected: 746 / 15000
Term: artificial intelligence | Page: 6 | Collected: 894 / 15000
Term: artificial intelligence | Page: 7 | Collected: 1030 / 15000
Term: artificial intelligence | Page: 8 | Collected: 1182 / 15000
Term: artificial intelligence | Page: 9 | Collected: 1321 / 15000
Term: artificial intelligence | Page: 10 | Collected: 1466 / 15000
Term: artificial intelligence | Page: 11 | Collected: 1607 / 15000
Term: artificial intelligence | Page: 12 | Collected: 1755 / 15000
Term: artificial intelligence | Page: 13 | Collected: 1906 / 15000
Term: artificial intelligence | Page: 14 | Collected: 2048 / 15000
Term: art

In [12]:
# Save raw OpenAlex response.
# This is useful in case you want to reprocess later without crawling again.

with open(RAW_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(all_papers, f, ensure_ascii=False, indent=2)

print("Saved raw JSON to:", RAW_JSON_FILE)

Saved raw JSON to: c:\Users\Nguyen Phuong Thao\Project_2_Data_Visualization_Group_6\Dataset\ai_papers_raw_15k.json


In [13]:
# Flatten nested OpenAlex JSON into a clean CSV table.
# This table is easier to use for EDA, dashboard, and visualization.

rows = []

for paper in all_papers:
    authors = []
    institutions = []
    countries = []

    # Extract authors, institutions, and countries.
    for auth in safe_list(paper.get("authorships")):
        author_name = safe_get(auth, ["author", "display_name"])
        if author_name:
            authors.append(author_name)

        for c in safe_list(auth.get("countries")):
            countries.append(c)

        for inst in safe_list(auth.get("institutions")):
            inst_name = inst.get("display_name")
            inst_country = inst.get("country_code")

            if inst_name:
                institutions.append(inst_name)

            if inst_country:
                countries.append(inst_country)

    # Extract topics.
    topics = []
    for topic in safe_list(paper.get("topics")):
        topic_name = topic.get("display_name")
        if topic_name:
            topics.append(topic_name)

    # Citation metrics.
    publication_year = paper.get("publication_year")
    citation_count = paper.get("cited_by_count") or 0

    if publication_year:
        paper_age = max(1, CURRENT_YEAR - int(publication_year) + 1)
        citations_per_year = citation_count / paper_age
    else:
        citations_per_year = None

    # Store one clean row per paper.
    rows.append({
        "paper_id": paper.get("id"),
        "title": paper.get("title") or paper.get("display_name"),
        "publication_year": publication_year,
        "publication_type": paper.get("type"),

        "citation_count": citation_count,
        "citations_per_year": citations_per_year,
        "referenced_works_count": paper.get("referenced_works_count"),

        "topics": join_unique(topics),
        "primary_topic": safe_get(paper, ["primary_topic", "display_name"]),
        "primary_subfield": safe_get(paper, ["primary_topic", "subfield", "display_name"]),
        "primary_field": safe_get(paper, ["primary_topic", "field", "display_name"]),
        "venue_source": safe_get(paper, ["primary_location", "source", "display_name"]),

        "authors": join_unique(authors),
        "institutions": join_unique(institutions),
        "countries": join_unique(countries),

        "doi": paper.get("doi"),
        "language": paper.get("language"),
        "is_oa": safe_get(paper, ["open_access", "is_oa"]),
        "oa_status": safe_get(paper, ["open_access", "oa_status"]),

        # This tells us which keyword first found the paper.
        "search_term_used": search_term_map.get(paper.get("id")),
    })

df = pd.DataFrame(rows)

# Remove duplicate papers again for safety.
df = df.drop_duplicates(subset=["paper_id"])

# Sort by year and citation count.
df = df.sort_values(
    by=["publication_year", "citation_count"],
    ascending=[True, False]
)

# Save clean CSV.
df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")

print("Saved clean CSV to:", CSV_FILE)
print("Dataset shape:", df.shape)

df.head()

Saved clean CSV to: c:\Users\Nguyen Phuong Thao\Project_2_Data_Visualization_Group_6\Dataset\ai_papers_clean_15k.csv
Dataset shape: (15000, 20)


,paper_id,title,publication_year,publication_type,citation_count,citations_per_year,referenced_works_count,topics,primary_topic,primary_subfield,primary_field,venue_source,authors,institutions,countries,doi,language,is_oa,oa_status,search_term_used
7467,https://openalex.org/W2089402107,Dynamic capabilities: what are they?,2000,article,14369,532.185185,118,Business Strategy and Innovation; Capital Inve...,Innovation and Knowledge Management,Strategy and Management,"Business, Management and Accounting",Strategic Management Journal,Jeffrey A. Martin; Kathleen M. Eisenhardt,Stanford University,US,https://doi.org/10.1002/1097-0266(200010/11)21...,en,True,bronze,machine learning
7518,https://openalex.org/W1534477342,Ensemble Methods in Machine Learning,2000,book-chapter,7783,288.259259,25,Face and Expression Recognition; Machine Learn...,Neural Networks and Applications,Artificial Intelligence,Computer Science,Lecture notes in computer science,Thomas G. Dietterich,Oregon State University,US,https://doi.org/10.1007/3-540-45014-9_1,en,False,closed,machine learning
12994,https://openalex.org/W2147111138,"Increasing Returns, Path Dependence, and the S...",2000,article,7613,281.962963,97,Economic Theory and Institutions; Historical E...,Economic Theory and Institutions,Economics and Econometrics,"Economics, Econometrics and Finance",American Political Science Review,Paul Pierson,Harvard University Press,US,https://doi.org/10.2307/2586011,en,True,green,deep learning
30,https://openalex.org/W2024046085,Additive logistic regression: a statistical vi...,2000,article,6931,256.703704,42,Advanced Statistical Methods and Models; Imbal...,Advanced Statistical Methods and Models,Statistics and Probability,Mathematics,The Annals of Statistics,Jerome H. Friedman; Robert Tibshirani; Trevor ...,Stanford University,US,https://doi.org/10.1214/aos/1016218223,en,True,bronze,artificial intelligence
38,https://openalex.org/W1608836379,Structural Equation Modeling and Regression: G...,2000,article,6358,235.481481,85,Big Data and Business Intelligence; Complex Sy...,Big Data and Business Intelligence,Management Information Systems,"Business, Management and Accounting",Communications of the Association for Informat...,David Gefen; Detmar W. Straub; Marie‐Claude Bo...,Drexel University; Georgia State University; U...,US,https://doi.org/10.17705/1cais.00407,en,True,bronze,artificial intelligence


In [14]:
# Basic dataset information.
df.info()

<class 'pandas.DataFrame'>
Index: 15000 entries, 7467 to 474
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   paper_id                15000 non-null  str    
 1   title                   15000 non-null  str    
 2   publication_year        15000 non-null  int64  
 3   publication_type        15000 non-null  str    
 4   citation_count          15000 non-null  int64  
 5   citations_per_year      15000 non-null  float64
 6   referenced_works_count  15000 non-null  int64  
 7   topics                  15000 non-null  str    
 8   primary_topic           15000 non-null  str    
 9   primary_subfield        15000 non-null  str    
 10  primary_field           15000 non-null  str    
 11  venue_source            13342 non-null  str    
 12  authors                 15000 non-null  str    
 13  institutions            15000 non-null  str    
 14  countries               15000 non-null  str    
 15  

In [15]:
# Check how many papers came from each search term.
df["search_term_used"].value_counts()

search_term_used
artificial intelligence    7433
machine learning           5527
deep learning              2040
Name: count, dtype: int64

In [16]:
# Check missing values.
df.isna().sum().sort_values(ascending=False)

venue_source              1658
language                    20
paper_id                     0
title                        0
publication_type             0
publication_year             0
referenced_works_count       0
citation_count               0
topics                       0
primary_topic                0
primary_subfield             0
citations_per_year           0
primary_field                0
authors                      0
countries                    0
institutions                 0
doi                          0
is_oa                        0
oa_status                    0
search_term_used             0
dtype: int64

In [17]:
# Check publication-year distribution.
df["publication_year"].value_counts().sort_index()

publication_year
2000     175
2001     157
2002     181
2003     210
2004     265
2005     262
2006     271
2007     279
2008     342
2009     331
2010     374
2011     367
2012     394
2013     429
2014     551
2015     712
2016     894
2017    1201
2018    1492
2019    1613
2020    1764
2021    1093
2022     770
2023     637
2024     210
2025      25
2026       1
Name: count, dtype: int64

In [18]:
# Check missing values.
df.isna().sum().sort_values(ascending=False)

venue_source              1658
language                    20
paper_id                     0
title                        0
publication_type             0
publication_year             0
referenced_works_count       0
citation_count               0
topics                       0
primary_topic                0
primary_subfield             0
citations_per_year           0
primary_field                0
authors                      0
countries                    0
institutions                 0
doi                          0
is_oa                        0
oa_status                    0
search_term_used             0
dtype: int64